# Baseline: pretrained embeddings (all-MiniLM-L6-v2)


**Метрики:** graded nDCG (core=2, contextual=1) та MAP — рахуються в `ranking_metrics.py`.


1. `cosine` — cosine similarity між embedding посади та навички (zero-shot, без навчання);
2. `LR concat` — LogisticRegression на `[E_job ; E_skill]`;
3. `LR extended` — LogisticRegression на `[E_job ; E_skill ; |E_job − E_skill| ; E_job * E_skill ; cosine]`.

Скор для LR: `P(1) + 2·P(2)` (очікувана релевантність).
Оцінювання — на dev-наборі (`data/raw/TaskB/validation`), для кожної посади ранжується весь корпус (9 052 навичок).

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

from config import RANDOM_SEED
from ranking_metrics import evaluate_query, save_results

MODEL_NAME = "all-MiniLM-L6-v2"
VAL_DIR = Path("data/raw/TaskB/validation")
TRAIN_PATH = Path("data/processed/train.csv")

## 1. Дані для оцінювання
Для ранжування беремо **сирі файли dev** (`queries`, `corpus_elements`, `qrels.tsv`) напряму: нам потрібен весь корпус навичок, а не `test.csv` зі штучними негативами.
Назву навички беремо так само, як у `data_processing.ipynb` — перший аліас (`skill_aliases[0]`).

In [2]:
queries = pd.read_csv(VAL_DIR / "queries", sep="\t").rename(columns={"jobtitle": "job_name", "q_id": "job_id"})
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t").rename(columns={"c_id": "skill_id"})
corpus["skill_aliases"] = corpus["skill_aliases"].apply(ast.literal_eval)
corpus["skill_name"] = corpus["skill_aliases"].apply(lambda x: x[0])
qrels = pd.read_csv(VAL_DIR / "qrels.tsv", header=None, sep="\t", names=["job_id", "_", "skill_id", "label"])

print(f"queries: {len(queries)}, corpus: {len(corpus)}, qrels: {len(qrels)}")

# (посада) -> вектор оцінок по всьому корпусу: 2 = core, 1 = contextual, 0 = решта
skill_index = {sid: i for i, sid in enumerate(corpus["skill_id"])}
grades_by_job = {}
for r in qrels.itertuples(index=False):
    grades_by_job.setdefault(r.job_id, np.zeros(len(corpus)))[skill_index[r.skill_id]] = r.label

queries: 304, corpus: 9052, qrels: 56418


## 2. Embeddings
Готова модель Sentence Transformers, без донавчання. Вектори нормалізовані (довжина 1), тому cosine = скалярний добуток.
Кодуємо лише унікальні тексти вони багато разів повторюються.

In [3]:
model = SentenceTransformer(MODEL_NAME)


def encode_texts(texts):
    """Повертає словник {текст: вектор} для унікальних текстів."""
    unique = sorted(set(texts))
    vectors = model.encode(unique, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    return dict(zip(unique, vectors))


job_emb = encode_texts(queries["job_name"].tolist())
skill_map = encode_texts(corpus["skill_name"].tolist())
skill_emb = np.vstack([skill_map[s] for s in corpus["skill_name"]])  # (9052, 384)
print(skill_emb.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/140 [00:00<?, ?it/s]

(9052, 384)


In [4]:
def evaluate(score_fn):
    """score_fn(вектор посади) -> масив скорів по всьому корпусу. Повертає метрики по кожній посаді."""
    per_query = []
    for row in queries.itertuples(index=False):
        if row.job_id not in grades_by_job:
            continue
        scores = score_fn(job_emb[row.job_name])
        per_query.append(evaluate_query(scores, grades_by_job[row.job_id]))
    return per_query

## 3.cosine similarity (zero-shot)
Жодного навчання: скор = схожість вектора посади та вектора навички.

In [5]:
per_query = evaluate(lambda e_job: skill_emb @ e_job)
save_results(
    "cosine_minilm",
    "Zero-shot: cosine similarity між embedding посади та навички (all-MiniLM-L6-v2), без навчання.",
    per_query,
)

Збережено: results/cosine_minilm.json


{'description': 'Zero-shot: cosine similarity між embedding посади та навички (all-MiniLM-L6-v2), без навчання.',
 'ndcg_graded': 0.6154,
 'ndcg_binary': 0.6599,
 'map_binary': 0.143,
 'mrr_binary': 0.6206,
 'p@1': 0.4342,
 'p@5': 0.3829,
 'p@10': 0.3539}

## 4. LogisticRegression на embeddings
навчається лише класифікатор на `label` з `train.csv` (0 - irrelevant, 1 - contextual, 2 - core).

In [6]:
def make_features(E_job, E_skill, mode):
    if mode == "concat":
        return np.hstack([E_job, E_skill])
    if mode == "extended":
        cosine = np.sum(E_job * E_skill, axis=1, keepdims=True)
        return np.hstack([E_job, E_skill, np.abs(E_job - E_skill), E_job * E_skill, cosine])
    raise ValueError(mode)


train = pd.read_csv(TRAIN_PATH)
print("train:", len(train), "рядків; label:", train["label"].value_counts().sort_index().to_dict())

tr_job_map = encode_texts(train["job_name"].astype(str).tolist())
tr_skill_map = encode_texts(train["skill_name"].astype(str).tolist())
E_job_tr = np.vstack([tr_job_map[j] for j in train["job_name"]])
E_skill_tr = np.vstack([tr_skill_map[s] for s in train["skill_name"]])
y_train = train["label"].values

train: 164699 рядків; label: {0: 50000, 1: 52219, 2: 62480}


Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches:   0%|          | 0/207 [00:00<?, ?it/s]

In [7]:
DESCRIPTIONS = {
    "concat": "LogisticRegression на конкатенації [E_job ; E_skill] (all-MiniLM-L6-v2, без донавчання). Скор = P(1) + 2*P(2).",
    "extended": "LogisticRegression на [E_job ; E_skill ; |E_job-E_skill| ; E_job*E_skill ; cosine] (all-MiniLM-L6-v2). Скор = P(1) + 2*P(2).",
}

for mode in ["concat", "extended"]:
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
    clf.fit(make_features(E_job_tr, E_skill_tr, mode), y_train)

    def lr_score(e_job, clf=clf, mode=mode):
        feats = make_features(np.tile(e_job, (len(skill_emb), 1)), skill_emb, mode)
        proba = clf.predict_proba(feats)  # колонки: класи 0, 1, 2
        return proba[:, 1] + 2 * proba[:, 2]

    per_query = evaluate(lr_score)
    save_results(f"lr_{mode}_minilm", DESCRIPTIONS[mode], per_query)

Збережено: results/lr_concat_minilm.json
Збережено: results/lr_extended_minilm.json


## 5. Порівняння підходів
з усіх `results/*.json`

In [8]:
import json

rows = []
for p in sorted(Path("results").glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"approach": p.stem, **{k: v for k, v in d.items() if k != "description"}})

pd.DataFrame(rows).sort_values("ndcg_graded", ascending=False).reset_index(drop=True)

,approach,ndcg_graded,ndcg_binary,map_binary,mrr_binary,p@1,p@5,p@10
0,cosine_minilm,0.6154,0.6599,0.1430,0.6206,0.4342,0.3829,0.3539
1,lr_extended_minilm,0.5902,0.6262,0.1137,0.3011,0.1151,0.1526,0.1842
2,lr_concat_minilm,0.4856,0.5213,0.0371,0.0947,0.0099,0.0303,0.0461
